# Nested Pandas Time Series with Hyrax

This notebook generates a random nested light-curve parquet file and loads it with `NestedPandasDataset`, requesting nested fields directly with dot notation.

In [ ]:
from pathlib import Path
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import hyrax

rng = np.random.default_rng(42)
output_path = Path("./random_lightcurve.parquet")

n_objects = 10
n_points = 64
times = np.linspace(59500, 59600, n_points)

rows = []
for idx in range(n_objects):
    period = rng.uniform(8.0, 25.0)
    phase = rng.uniform(0, 2 * np.pi)
    flux = np.sin(times / period + phase) + rng.normal(0, 0.1, n_points)
    flux_err = rng.uniform(0.02, 0.12, n_points)
    band = rng.choice([0, 1, 2], size=n_points)
    rows.append(
        {
            "object_id": f"obj-{idx:04d}",
            "lightcurve": {
                "time": times.tolist(),
                "flux": flux.tolist(),
                "flux_err": flux_err.tolist(),
                "band": band.tolist(),
            },
        }
    )

table = pa.Table.from_pylist(rows)
pq.write_table(table, output_path)

h = hyrax.Hyrax()
h.config["data_request"] = {
    "infer": {
        "lc": {
            "dataset_class": "NestedPandasDataset",
            "data_location": str(output_path),
            "primary_id_field": "object_id",
            # Request only nested columns using nested-pandas dot-notation.
            "fields": ["lightcurve.time", "lightcurve.flux", "lightcurve.flux_err", "lightcurve.band"],
        }
    }
}

prepared = h.prepare()
print(f"Loaded rows: {len(prepared['infer'])}")

After loading, inspect `prepared['infer'][0].keys()` and set `fields` to the exact nested columns you want (for example `lightcurve.time`).

In [ ]:
prepared["infer"][0]["lc"].keys()

In [ ]:
prepared["infer"][0]["lc"]["lightcurve.time"][:5], prepared["infer"][0]["lc"]["lightcurve.flux"][:5]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

time = np.asarray(prepared["infer"][0]["lc"]["lightcurve.time"])
flux = np.asarray(prepared["infer"][0]["lc"]["lightcurve.flux"])
flux_err = np.asarray(prepared["infer"][0]["lc"]["lightcurve.flux_err"])
band = np.asarray(prepared["infer"][0]["lc"]["lightcurve.band"])

plt.figure(figsize=(10, 5))
for b in np.unique(band):
    m = band == b
    plt.errorbar(time[m], flux[m], flux_err[m], fmt="o", label=f"band {b}", alpha=0.8)

plt.xlabel("MJD")
plt.ylabel("Flux")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.legend()
plt.show()